# Daily CPTEC Obs (MERGE + SAMeT)

Landing diario de las observaciones en grilla de CPTEC/INPE, corriendo en Databricks serverless:
baja de `ftp.cptec.inpe.br` el dia anterior (y una ventana hacia atras) de **MERGE** (precipitacion
diaria 0,1 grado, acumulado 12Z(D-1)->12Z(D), publicado ~02:40 UTC de D+1) y **SAMeT** (TMAX/TMED/TMIN
diarias 0,05 grado, dia calendario UTC, publicadas ~03:10 UTC de D+1), recorta al bounding box de la
cuenca y escribe UN Parquet por producto y dia en `weather.raw.cptec_volume/{merge,samet}/daily/`
-- el mismo formato que produce el backfill local (`notebooks_local/cptec_obs/`), asi Bronze no
distingue de donde vino cada archivo.

**Ventana hacia atras (no solo D-1):** CPTEC regenera MERGE en los primeros dias del mes siguiente
(pluviometros completos) y SAMeT ~7 dias despues (ERA5). Cada dia se compara el `Last-Modified` HTTP
del archivo de origen con el `source_last_modified` del Parquet ya landeado y se re-descarga solo lo
que cambio. **MERGE viene en GRIB2 (empaquetado complejo) y se decodifica con `pygrib`**, no con
`cfgrib`/`eccodes` (abortan el kernel en serverless, Decision 013) -- `pygrib` verificado en este
workspace el 2026-08-26. Ver `docs/data_sources.md` §9.6/§9.7 y Decision 033.

In [ ]:
%pip install --quiet pygrib netCDF4 pyarrow requests numpy
dbutils.library.restartPython()

In [ ]:
import json
import math
import os
import shutil
import tempfile
from datetime import date, datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import requests

try:
    dbutils.widgets.text('merge_lookback_days', '45')
    dbutils.widgets.text('samet_lookback_days', '14')
    dbutils.widgets.dropdown('force_reload', 'false', ['false', 'true'])
    merge_lookback_days = int(dbutils.widgets.get('merge_lookback_days'))
    samet_lookback_days = int(dbutils.widgets.get('samet_lookback_days'))
    force_reload = dbutils.widgets.get('force_reload').lower() == 'true'
except Exception:
    merge_lookback_days, samet_lookback_days, force_reload = 45, 14, False

VOLUME_ROOT = Path('/Volumes/weather/raw/cptec_volume')
GEOJSON_PATH = '/Workspace/Users/joaquintschopp@gmail.com/rio-uruguay-hydro-pipeline/SIG/subcuencas_modelo.geojson'
MERGE_BASE_URL = 'https://ftp.cptec.inpe.br/modelos/tempo/MERGE/GPM/DAILY'
SAMET_BASE_URL = 'https://ftp.cptec.inpe.br/modelos/tempo/SAMeT/DAILY'
SAMET_VARS = ('TMED', 'TMAX', 'TMIN')
SAMET_FILL = -9.99e08
SOURCE_API = {'merge': 'cptec_merge_gpm_daily', 'samet': 'cptec_samet_daily'}
GRID_DEG = {'merge': 0.1, 'samet': 0.05}
LOOKBACK = {'merge': merge_lookback_days, 'samet': samet_lookback_days}
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'

print(f'merge_lookback_days={merge_lookback_days}, samet_lookback_days={samet_lookback_days}, force_reload={force_reload}')

In [ ]:
# Helpers replicados de notebooks_local/cptec_obs/common_cptec.py (los notebooks de Databricks no
# importan modulos del repo; mismo patron que Daily_ECMWF_CF.ipynb con common_ecmwf.py). Cualquier
# cambio de formato/recorte hay que hacerlo en los dos lados.


def _geojson_total_bounds(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    minx = miny = math.inf
    maxx = maxy = -math.inf

    def walk(coords):
        nonlocal minx, miny, maxx, maxy
        if isinstance(coords[0], (int, float)):
            minx, maxx = min(minx, coords[0]), max(maxx, coords[0])
            miny, maxy = min(miny, coords[1]), max(maxy, coords[1])
        else:
            for c in coords:
                walk(c)

    for feature in data['features']:
        walk(feature['geometry']['coordinates'])
    return minx, miny, maxx, maxy


def compute_download_area(grid_deg, margin_cells=1):
    minx, miny, maxx, maxy = _geojson_total_bounds(GEOJSON_PATH)
    margin = grid_deg * margin_cells
    return {
        'north': round(math.ceil((maxy + margin) / grid_deg) * grid_deg, 4),
        'south': round(math.floor((miny - margin) / grid_deg) * grid_deg, 4),
        'west': round(math.floor((minx - margin) / grid_deg) * grid_deg, 4),
        'east': round(math.ceil((maxx + margin) / grid_deg) * grid_deg, 4),
    }


def normalize_longitude(lon):
    return lon - 360 if lon > 180 else lon


def merge_url(d):
    return f'{MERGE_BASE_URL}/{d:%Y}/{d:%m}/MERGE_CPTEC_{d:%Y%m%d}.grib2'


def samet_url(d, var):
    return f'{SAMET_BASE_URL}/{var}/{d:%Y}/{d:%m}/SAMeT_CPTEC_{var}_{d:%Y%m%d}.nc'


def parquet_name(source, d):
    return f'{source.upper()}_{d:%Y_%m_%d}.parquet'


def _parse_last_modified(headers):
    raw = headers.get('Last-Modified')
    if not raw:
        return None
    try:
        return parsedate_to_datetime(raw).astimezone(timezone.utc)
    except (TypeError, ValueError):
        return None


session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT})


def head_last_modified(url):
    resp = session.head(url, timeout=60, allow_redirects=True)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return _parse_last_modified(resp.headers)


def fetch(url, max_retries=4):
    import time
    for attempt in range(max_retries):
        try:
            resp = session.get(url, timeout=120)
            if resp.status_code == 404:
                return None, None
            resp.raise_for_status()
            return resp.content, _parse_last_modified(resp.headers)
        except (requests.exceptions.RequestException, OSError):
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)
    return None, None


class Grid2D:
    def __init__(self, lat, lon, values, extra):
        self.lat = np.asarray(lat, dtype='float64')
        self.lon = np.asarray(lon, dtype='float64')
        self.values = np.asarray(values, dtype='float64')
        self.extra = None if extra is None else np.asarray(extra, dtype='float64')


def decode_merge_pygrib(content):
    # Mensaje 1 = precipitacion (CPTEC lo etiqueta 'rdp'), mensaje 2 = NEST (pluviometros por
    # punto, etiquetado 'prmsl'). Los faltantes vienen por missing-value-management del
    # empaquetado complejo, no por bitmap: se reemplaza explicitamente `missingValue` por NaN.
    import pygrib
    tmp = os.path.join(tempfile.gettempdir(), f'merge_{os.getpid()}.grib2')
    with open(tmp, 'wb') as f:
        f.write(content)
    grbs = pygrib.open(tmp)
    try:
        msgs = [grbs.message(i) for i in range(1, grbs.messages + 1)]
        if not msgs:
            raise ValueError('GRIB2 de MERGE sin mensajes')
        fields = []
        for m in msgs:
            vals = np.ma.filled(np.ma.asarray(m.values).astype('float64'), np.nan)
            try:
                missing = float(m['missingValue'])
                vals = np.where(vals == missing, np.nan, vals)
            except Exception:
                pass
            fields.append(vals)
        lats, lons = msgs[0].latlons()
        lat = np.asarray(lats)[:, 0]
        lon = np.array([normalize_longitude(float(x)) for x in np.asarray(lons)[0, :]])
    finally:
        grbs.close()
        try:
            os.remove(tmp)
        except OSError:
            pass
    return Grid2D(lat, lon, fields[0], fields[1] if len(fields) > 1 else None)


def decode_samet_netcdf(content, var):
    import netCDF4
    ds = netCDF4.Dataset('inmemory.nc', mode='r', memory=content)
    try:
        lat = np.asarray(ds.variables['lat'][:], dtype='float64')
        lon = np.asarray(ds.variables['lon'][:], dtype='float64')
        main = np.ma.filled(ds.variables[var.lower()][:].astype('float64'), np.nan)
        nobs = np.ma.filled(ds.variables['nobs'][:].astype('float64'), np.nan) if 'nobs' in ds.variables else None
    finally:
        ds.close()
    main = np.squeeze(main)
    main = np.where(main <= SAMET_FILL * 0.5, np.nan, main)
    if nobs is not None:
        nobs = np.squeeze(nobs)
        nobs = np.where(nobs <= SAMET_FILL * 0.5, np.nan, nobs)
    return Grid2D(lat, lon, main, nobs)


def crop_indices(lat, lon, area):
    ilat = np.where((lat >= area['south'] - 1e-6) & (lat <= area['north'] + 1e-6))[0]
    ilon = np.where((lon >= area['west'] - 1e-6) & (lon <= area['east'] + 1e-6))[0]
    if len(ilat) == 0 or len(ilon) == 0:
        raise ValueError(f'El recorte al area {area} no deja ningun punto de grilla')
    return ilat, ilon


def flatten_merge(fecha, grid, area, source_file, source_last_modified, extracted_at):
    ilat, ilon = crop_indices(grid.lat, grid.lon, area)
    sub = grid.values[np.ix_(ilat, ilon)]
    nest = grid.extra[np.ix_(ilat, ilon)] if grid.extra is not None else np.full(sub.shape, np.nan)
    lon2d, lat2d = np.meshgrid(np.round(grid.lon[ilon], 3), np.round(grid.lat[ilat], 3))
    keep = ~np.isnan(sub)
    n = int(keep.sum())
    nest_col = nest[keep]
    return pa.table({
        'fecha': pa.array([fecha] * n, type=pa.date32()),
        'latitude': pa.array(lat2d[keep], type=pa.float64()),
        'longitude': pa.array(lon2d[keep], type=pa.float64()),
        'prec_mm': pa.array(sub[keep], type=pa.float64()),
        'nest': pa.array(np.where(np.isnan(nest_col), 0, nest_col).astype('int32'), type=pa.int32()),
        'source_file': pa.array([source_file] * n, type=pa.string()),
        'source_last_modified': pa.array([source_last_modified] * n, type=pa.timestamp('us', tz='UTC')),
        'source_api': pa.array([SOURCE_API['merge']] * n, type=pa.string()),
        'extracted_at': pa.array([extracted_at] * n, type=pa.timestamp('us', tz='UTC')),
    })


def flatten_samet(fecha, grids, area, source_files, last_modified, extracted_at):
    ref = next((g for g in grids.values() if g is not None), None)
    if ref is None:
        raise ValueError('SAMeT: ningun archivo disponible para el dia')
    ilat, ilon = crop_indices(ref.lat, ref.lon, area)
    lon2d, lat2d = np.meshgrid(np.round(ref.lon[ilon], 3), np.round(ref.lat[ilat], 3))
    cols, nobs_cols = {}, {}
    any_valid = np.zeros(lat2d.shape, dtype=bool)
    for var in SAMET_VARS:
        g = grids.get(var)
        if g is None:
            cols[var] = np.full(lat2d.shape, np.nan)
            nobs_cols[var] = np.full(lat2d.shape, np.nan)
            continue
        cols[var] = g.values[np.ix_(ilat, ilon)]
        nobs_cols[var] = g.extra[np.ix_(ilat, ilon)] if g.extra is not None else np.full(lat2d.shape, np.nan)
        any_valid |= ~np.isnan(cols[var])
    keep = any_valid
    n = int(keep.sum())
    present = [v for v in SAMET_VARS if grids.get(v) is not None]
    lm_values = [last_modified[v] for v in present if last_modified.get(v) is not None]
    lm = max(lm_values) if lm_values else None

    def nobs_col(var):
        arr = nobs_cols[var][keep]
        return pa.array(np.where(np.isnan(arr), 0, arr).astype('int32'), type=pa.int32())

    return pa.table({
        'fecha': pa.array([fecha] * n, type=pa.date32()),
        'latitude': pa.array(lat2d[keep], type=pa.float64()),
        'longitude': pa.array(lon2d[keep], type=pa.float64()),
        'tmed_c': pa.array(cols['TMED'][keep], type=pa.float64()),
        'tmax_c': pa.array(cols['TMAX'][keep], type=pa.float64()),
        'tmin_c': pa.array(cols['TMIN'][keep], type=pa.float64()),
        'nobs_tmed': nobs_col('TMED'),
        'nobs_tmax': nobs_col('TMAX'),
        'nobs_tmin': nobs_col('TMIN'),
        'source_file': pa.array([','.join(source_files[v] for v in present)] * n, type=pa.string()),
        'source_last_modified': pa.array([lm] * n, type=pa.timestamp('us', tz='UTC')),
        'source_api': pa.array([SOURCE_API['samet']] * n, type=pa.string()),
        'extracted_at': pa.array([extracted_at] * n, type=pa.timestamp('us', tz='UTC')),
    })


def write_parquet_to_volume(table, dest):
    # Se escribe a un temporal local y se copia entero: un Parquet a medias en el Volume seria
    # leido por Bronze como archivo corrupto.
    tmp = os.path.join(tempfile.gettempdir(), dest.name + '.tmp')
    pq.write_table(table, tmp, compression='zstd')
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, dest)
    os.remove(tmp)


def landed_last_modified(dest):
    if not dest.exists():
        return None, False
    try:
        col = pq.read_table(dest, columns=['source_last_modified']).column(0)
        return (col[0].as_py() if len(col) else None), True
    except Exception:
        return None, True

In [ ]:
def land_merge(d, area):
    dest = VOLUME_ROOT / 'merge' / 'daily' / parquet_name('merge', d)
    url = merge_url(d)
    landed_lm, exists = landed_last_modified(dest)
    if exists and not force_reload:
        remote_lm = head_last_modified(url)
        if remote_lm is None:
            return 'not_found'
        if landed_lm is not None and remote_lm <= landed_lm:
            return 'unchanged'
    content, remote_lm = fetch(url)
    if content is None:
        return 'not_found'
    grid = decode_merge_pygrib(content)
    table = flatten_merge(d, grid, area, f'MERGE_CPTEC_{d:%Y%m%d}.grib2', remote_lm, datetime.now(timezone.utc))
    write_parquet_to_volume(table, dest)
    return 'updated' if exists else 'new'


def land_samet(d, area):
    dest = VOLUME_ROOT / 'samet' / 'daily' / parquet_name('samet', d)
    landed_lm, exists = landed_last_modified(dest)
    urls = {var: samet_url(d, var) for var in SAMET_VARS}
    if exists and not force_reload:
        remote = {var: head_last_modified(u) for var, u in urls.items()}
        present = [lm for lm in remote.values() if lm is not None]
        if not present:
            return 'not_found'
        if landed_lm is not None and max(present) <= landed_lm and all(lm is not None for lm in remote.values()):
            return 'unchanged'
    grids, files, lms = {}, {}, {}
    for var, u in urls.items():
        content, lm = fetch(u)
        if content is None:
            grids[var] = None
            continue
        grids[var] = decode_samet_netcdf(content, var)
        files[var] = f'SAMeT_CPTEC_{var}_{d:%Y%m%d}.nc'
        lms[var] = lm
    if all(g is None for g in grids.values()):
        return 'not_found'
    table = flatten_samet(d, grids, area, files, lms, datetime.now(timezone.utc))
    write_parquet_to_volume(table, dest)
    if any(g is None for g in grids.values()):
        return 'partial'
    return 'updated' if exists else 'new'


summary = {}
yesterday = date.today() - timedelta(days=1)
for source, fn in (('merge', land_merge), ('samet', land_samet)):
    area = compute_download_area(GRID_DEG[source])
    counts = {}
    start = yesterday - timedelta(days=LOOKBACK[source])
    print(f'[{source}] ventana {start} .. {yesterday}, area {area}')
    d = start
    while d <= yesterday:
        try:
            status = fn(d, area)
        except Exception as exc:
            status = 'failed'
            print(f'  [{source} {d}] FALLO: {type(exc).__name__}: {str(exc)[:200]}')
        counts[status] = counts.get(status, 0) + 1
        if status in ('new', 'updated', 'partial', 'failed'):
            print(f'  [{source} {d}] {status}')
        d += timedelta(days=1)
    summary[source] = counts
    print(f'[{source}] {counts}')

# Freshness: el dato de ayer debe estar (MERGE ~02:40 UTC, SAMeT ~03:10 UTC de D+1). Si no
# esta, se avisa pero no se aborta: el job de Gold no depende de esta fuente (Decision 033).
for source in ('merge', 'samet'):
    dest = VOLUME_ROOT / source / 'daily' / parquet_name(source, yesterday)
    print(f'[{source}] {yesterday}: {"OK" if dest.exists() else "TODAVIA NO DISPONIBLE"}')

dbutils.notebook.exit(json.dumps(summary))